# Topic H — Longstaff-Schwartz (LSM) for American Options

*FINM 32000 Numerical Methods — 15-minute exam-ready version*

---

## How to read this notebook

**Goal**: be able to answer Practice-Final P5 (3 bug fixes) **perfectly** and write the HW7 P1 LSM code from memory.

1. Section 1 — **The recipe** (~3 min): 5 plain-English steps + code skeleton.
2. Section 2 — **Tiny worked example** (~3 min): 4 paths, 3 dates, every number by hand.
3. Section 3 — **The 3 exam bugs** (~3 min): Practice-Final P5 side-by-side.
4. Section 4 — **Full runnable code** (~2 min): copy-paste template.
5. Section 5 — **Pitfalls + exercises** (~4 min).

Skip the appendix unless you have time.

**Variable names below = exactly what you'd type in code = what the exam uses.** No fancy notation.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)


## 1. The recipe (memorize this)

**Goal**: price a Bermudan put with $N$ exercise dates $t_1, t_2, \ldots, t_N$, using $M$ simulated paths.

### The big picture in one sentence

> *"Walk backward in time. At each exercise date, fit a curve through the cloud of (today's stock price, future cashflow if you held). If today's exercise value beats that fitted curve, exercise; else hold."*

### The 5 steps

**Step 1 — Simulate $M$ paths.**

```python
paths = S0 * exp((r - 0.5*sigma**2) * dt * [1,2,...,N]   # drift
              + sigma * sqrt(dt) * cumsum(Z, axis=1))    # diffusion
# shape: (M, N).  paths[m, n] = stock price on path m at time t_{n+1}
```

**Step 2 — Initialize at expiry (the last column).**

```python
payoffDiscounted = maximum(0, K - paths[:, -1])
# At time T, exercise if ITM. No future to compare against.
```

`payoffDiscounted` will hold the **discounted-to-current-time cashflow on each path** as we sweep backward.

**Step 3 — Loop backward from $n = N-1, N-2, \ldots, 1$.**

At each time $t_n$:

```python
# 3a. Discount the *future* cashflow back to t_n
continuationPayoffDiscounted = exp(-r*dt) * payoffDiscounted

# 3b. State variable today, and today's exercise value
X = paths[:, n-1]
exerciseValue = K - X

# 3c. Fit a curve.  Use ITM paths only (HW7 hint)
basis = stack([1, X, X**2], axis=1)         # M-by-3
mask = exerciseValue > 0
coeffs = lstsq(basis[mask], continuationPayoffDiscounted[mask])[0]
estimatedContinuationValue = basis @ coeffs   # smooth curve evaluated on all paths

# 3d. Decide: exercise if today beats the fitted curve
whichPathsToExercise = (exerciseValue >= maximum(estimatedContinuationValue, 0))

# 3e. Update payoffDiscounted
payoffDiscounted = where(whichPathsToExercise,
                         exerciseValue,                  # we exercised
                         continuationPayoffDiscounted)   # we held
```

**Step 4 — Handle time 0 (no regression).**

```python
# Discount once more, average across paths, but ALSO check time-0 exercise
putprice = max(K - S0,                              # exercise now
               exp(-r*dt) * payoffDiscounted.mean()) # average value of holding
```

**Step 5 — Return `putprice`. Done.**

---

### Why each piece does what it does

| Variable | What it holds |
|---|---|
| `paths[m, n]` | Stock price on path $m$ at the $n$-th exercise date |
| `payoffDiscounted` (after step 3e) | What each path "earns," discounted to time $t_n$, **under optimal play from $t_n$ onward** |
| `continuationPayoffDiscounted` | Same thing, discounted **one more step back** to $t_{n-1}$. It is what you would get **if you held at $t_{n-1}$** |
| `exerciseValue` | $K - X$, what you would get if you exercised at $t_{n-1}$ |
| `basis` | The features $[1, X, X^2]$ — pick a parametric form |
| `coeffs` | OLS regression weights $\hat\beta_0, \hat\beta_1, \hat\beta_2$ |
| `estimatedContinuationValue` | The **fitted curve** evaluated on all paths — our estimate of $\mathbb E[\text{future cashflow} \mid X]$ |
| `whichPathsToExercise` | Boolean per path: True if exercise beats hold |

**The single most important conceptual point**: at step 3d we compare exercise to the **fitted curve**, not to `continuationPayoffDiscounted` itself. The latter is one noisy realization; the former is our estimate of its conditional expectation. **Practice-Final P5 Bug 2 is exactly this confusion.**


## 2. Tiny worked example — 4 paths, 3 dates, by hand

Let's price a Bermudan put with **everything tiny** so we can do every step by hand.

- $S_0 = 10$, $K = 11$, $r = 0$, $\sigma = 0.2$, $T = 3$, $N = 3$ exercise dates (so $dt = 1$).
- $M = 4$ paths.

After simulating, suppose the paths come out:

```
path m | t=1     t=2     t=3
-------+---------------------
   1   |  9       8      7
   2   | 10       9      8
   3   | 11      12     13
   4   | 12      13     14
```

(Real LSM uses $M$ in the thousands. This is purely for intuition.)

### Step 2 — initialize at expiry ($t=3$)

```
payoffDiscounted = max(0, 11 - [7, 8, 13, 14]) = [4, 3, 0, 0]
```

Path 1 finishes deep ITM → payoff 4. Path 3, 4 finish OTM → payoff 0.

### Step 3 — backward step from $t=3$ to $t=2$

`r=0` so no discounting matters; `continuationPayoffDiscounted = payoffDiscounted = [4, 3, 0, 0]`.

State at $t=2$: `X = [8, 9, 12, 13]`. `exerciseValue = K - X = [3, 2, -1, -2]`.

Only paths 1 and 2 are **ITM** at $t=2$. Restrict regression to those:

```
Regress  [4, 3]  on basis  [[1, 8, 64], [1, 9, 81]]
```

With only 2 points and 3 unknowns, the system is underdetermined. In practice you would have many ITM paths; here we use the simplest fit (pretend $\hat f(X) = 12 - X$ from a linear fit, ignoring quadratic). Then on all 4 paths:

```
estimatedContinuationValue ≈ [12-8, 12-9, 12-12, 12-13] = [4, 3, 0, -1]
                          → max(., 0) = [4, 3, 0, 0]
```

Compare to exerciseValue = [3, 2, -1, -2]:

```
exerciseValue >= max(estContValue, 0):
  path 1: 3 >= 4 ?  No   -> hold
  path 2: 2 >= 3 ?  No   -> hold
  path 3: -1 >= 0?  No   -> hold (and exerciseValue is negative anyway, but also masked)
  path 4: -2 >= 0?  No   -> hold

whichPathsToExercise = [False, False, False, False]
```

So no path exercises at $t=2$. The cashflows stay at `[4, 3, 0, 0]`.

### Step 3 — backward step from $t=2$ to $t=1$

`continuationPayoffDiscounted = [4, 3, 0, 0]` (still, since `r=0`).

State at $t=1$: `X = [9, 10, 11, 12]`. `exerciseValue = K - X = [2, 1, 0, -1]`.

ITM paths: only paths 1, 2 (with exerciseValue $> 0$). Path 3 has `exerciseValue = 0`, technically on the boundary; conventionally excluded.

Regress `[4, 3]` on `[[1, 9, 81], [1, 10, 100]]` (same kind of underdetermined). Suppose the fit gives `estimatedContinuationValue ≈ [4, 3, 2, 1]` (a smooth interpolation).

Compare:

```
  path 1: exercise 2 vs fitted 4 ?  hold
  path 2: exercise 1 vs fitted 3 ?  hold
  path 3: exercise 0 vs fitted 2 ?  hold (also boundary)
  path 4: exercise -1, masked      hold
```

Still nobody exercises. `payoffDiscounted = [4, 3, 0, 0]`.

### Step 4 — time 0

```
continuation = mean([4, 3, 0, 0]) = 1.75
exercise_now = K - S0 = 11 - 10 = 1
putprice = max(1, 1.75) = 1.75
```

### What this tells you about the exam

- The **mechanics are simple**: build basis, fit, compare, update.
- The **regression keeps the cashflows alive**, even if many paths never exercise.
- The **time-0 max** matters: if `K - S0 = 1` had been bigger than the continuation `1.75`, the answer would have been the exercise value. **This is Practice-Final P5 Bug 3.**


## 3. The 3 Practice-Final P5 bugs (the heart of the exam)

The exam shows you LSM code with 3 bugs in `price_americanPut_GBM`. **Memorize this table.**

---

### Bug 1 — wrong drift in `paths = ...`

| | Code |
|---|---|
| **Wrong** | `paths = S0 * np.exp(r*dt*np.tile(...) + sigma*np.sqrt(dt)*np.cumsum(Z, axis=1))` |
| **Right** | `paths = S0 * np.exp((r - 0.5*sigma**2)*dt*np.tile(...) + sigma*np.sqrt(dt)*np.cumsum(Z, axis=1))` |
| **Why** | Itô correction. For GBM, $\log S_t$ has drift $r - \tfrac12\sigma^2$, not $r$. Without the correction, $\mathbb E[S_t] = S_0 e^{(r+\sigma^2/2)t}$, biased high. |
| **Where else** | Practice-Final P3 Bug 3 (the dividend problem). Same idea, different setting. |

---

### Bug 2 — comparing exercise to noisy continuation

| | Code |
|---|---|
| **Wrong** | `whichPathsToExercise = (exerciseValue >= np.maximum(continuationPayoffDiscounted, 0))` |
| **Right** | `whichPathsToExercise = (exerciseValue >= np.maximum(estimatedContinuationValue, 0))` |
| **Why** | `continuationPayoffDiscounted` is **one noisy realization** of what happens if you hold. `estimatedContinuationValue` is the **fitted curve** — our estimate of the conditional expectation. We act on expectations, not luck. |
| **Visualization** | In Excel/LSM literature: the **blue scatter** is `continuationPayoffDiscounted`. The **red curve** is `estimatedContinuationValue`. Compare exercise to red, not blue. |

---

### Bug 3 — missing time-0 exercise check

| | Code |
|---|---|
| **Wrong** | `putprice = np.exp(-r*dt) * np.mean(payoffDiscounted)` |
| **Right** | `putprice = max(K - S0, np.exp(-r*dt) * np.mean(payoffDiscounted))` |
| **Why** | At $t = 0$ there's no regression (only 1 point), but the holder can still **exercise immediately**. For deep ITM puts ($S_0 \ll K$), immediate exercise can dominate. Missing this max under-prices the option. |

---

### Diagnosing on the exam

When you see LSM code on the exam:
1. **Scan the `paths =` line** for `r*dt` vs `(r - 0.5*sigma**2)*dt`. → Bug 1.
2. **Scan `whichPathsToExercise =`** for `continuationPayoffDiscounted` vs `estimatedContinuationValue`. → Bug 2.
3. **Scan the final `putprice =`** for a `max(K - S0, ...)`. → Bug 3.

If a line says "FILL THIS IN" but is provided correctly elsewhere — that's also a hint.


## 4. Full runnable code (HW7 P1 reference)

This is the template you should be able to **write from scratch on the exam**. Memorize the structure.


In [2]:
def price_americanPut_GBM(S0, K, T, r, sigma, M=10_000, N=4, seed=0):
    rng = np.random.default_rng(seed)
    dt = T / N
    Z = rng.normal(size=(M, N))

    # Step 1: simulate paths (Bug 1 lives here)
    time_idx = np.tile(np.arange(1, N + 1), (M, 1))
    paths = S0 * np.exp((r - 0.5 * sigma**2) * dt * time_idx
                        + sigma * np.sqrt(dt) * np.cumsum(Z, axis=1))

    # Step 2: initialize at expiry
    payoffDiscounted = np.maximum(0.0, K - paths[:, -1])

    # Step 3: backward induction
    for n in range(N - 1, 0, -1):
        continuationPayoffDiscounted = np.exp(-r * dt) * payoffDiscounted

        X = paths[:, n - 1]
        exerciseValue = K - X

        basis = np.stack((np.ones(M), X, X**2), axis=1)

        mask_ITM = exerciseValue > 0
        if mask_ITM.sum() > 3:
            coeffs, *_ = np.linalg.lstsq(basis[mask_ITM],
                                         continuationPayoffDiscounted[mask_ITM],
                                         rcond=None)
            estimatedContinuationValue = basis @ coeffs
        else:
            estimatedContinuationValue = np.zeros(M)

        # Bug 2 lives here: must compare to estimatedContinuationValue (fitted)
        whichPathsToExercise = (exerciseValue >= np.maximum(estimatedContinuationValue, 0)) & mask_ITM

        payoffDiscounted = np.where(whichPathsToExercise,
                                    exerciseValue,
                                    continuationPayoffDiscounted)

    # Step 4: time-0 (Bug 3 lives here: must include max with immediate exercise)
    continuation_at_0 = np.exp(-r * dt) * payoffDiscounted.mean()
    putprice = max(K - S0, continuation_at_0)
    return putprice


# HW7 P1 parameters
price = price_americanPut_GBM(S0=1.0, K=1.1, T=4.0, r=0.03, sigma=0.20,
                              M=10_000, N=4, seed=0)
print(f"HW7 P1 Bermudan put price: {price:.4f}  (HW7 IA reference: ~0.16)")

# Run a few seeds to see the MC noise
prices = [price_americanPut_GBM(1.0, 1.1, 4.0, 0.03, 0.20, M=10_000, N=4, seed=s)
          for s in range(10)]
print(f"Across 10 seeds: mean {np.mean(prices):.4f}, std {np.std(prices, ddof=1):.4f}")


HW7 P1 Bermudan put price: 0.1621  (HW7 IA reference: ~0.16)
Across 10 seeds: mean 0.1630, std 0.0012


## 5. Common pitfalls (one-line each)

| # | Pitfall | Fix |
|---|---|---|
| 1 | Drift uses `r` not `(r - 0.5*sigma**2)` in `paths` | Use the Itô-corrected drift |
| 2 | Compare exercise to `continuationPayoffDiscounted` instead of `estimatedContinuationValue` | Compare to the **fitted** curve |
| 3 | `putprice = exp(-r*dt) * payoffDiscounted.mean()` (no time-0 max) | `max(K - S0, exp(-r*dt) * payoffDiscounted.mean())` |
| 4 | Regressing on **all** paths instead of ITM-only | Use `mask_ITM = exerciseValue > 0` |
| 5 | Forgetting to clip negative fitted values | Use `np.maximum(estimatedContinuationValue, 0)` |
| 6 | `np.linalg.lstsq` returns a tuple — using `coeffs` directly fails | Unpack with `coeffs, *_ = np.linalg.lstsq(...)` |
| 7 | Updating `payoffDiscounted` in-place and losing the "hold" branch | Use `np.where(condition, exercise, hold)` |
| 8 | Skipping `& mask_ITM` in `whichPathsToExercise` | Add it as a safety belt — never exercise OTM puts |

### How `payoffDiscounted` flows through the loop

At any point in the backward loop, `payoffDiscounted` represents:

> *"What this path earns under our **current best** stopping policy, discounted back to **the current time** in the loop."*

After step 3e at time $t_n$, it's discounted to $t_n$. At the next iteration's step 3a, we multiply by $e^{-r\,dt}$ to bring it back to $t_{n-1}$, then compare to today's exercise. This is why we **reuse the same 1-D array** for every time step.


## 6. Exam-style exercises

**E1.** In Practice-Final P5, what would happen to the reported put price if **only Bug 1** (wrong drift) were left uncorrected?

**E2.** In Practice-Final P5, what would happen to the reported put price if **only Bug 3** (missing time-0 max) were left uncorrected, for a put with $K = 1.1$ and $S_0 = 0.5$?

**E3.** In the regression step, why do we use $[1, X, X^2]$ as basis functions and not just $[X]$?

**E4.** Why is **ITM-only regression** (L7.12) a good idea, and when might it backfire?

**E5.** You see code with `whichPathsToExercise = (exerciseValue >= estimatedContinuationValue)` (no `np.maximum(., 0)`). Under what conditions does the missing `max` create a bug?


### Solutions

<details><summary><b>E1</b></summary>

The drift would be too high (uses $r$ instead of $r - \sigma^2/2$). Stock paths drift up faster than they should, so the put is **less likely** to be ITM and the price is **biased low**.

For HW7 parameters with $\sigma = 0.20$, the missing drift correction is $\sigma^2/2 = 0.02$, applied over $T = 4$, so $\log S_T$ shifts up by $0.08$. The OTM bias is small here but real — the answer would land notably below the true 0.16.
</details>

<details><summary><b>E2</b></summary>

Immediate exercise gives $K - S_0 = 1.1 - 0.5 = 0.6$. The continuation value (held to $t_1$) is at most $K \cdot e^{-r\,dt} \approx 1.1$, but it can also be substantially less because of the chance of finishing OTM. For very deep ITM, exercise-now dominates, so missing the `max` could under-price by **a lot** — potentially returning a value less than the **intrinsic** value, which is impossible for an American option.

**Key insight:** an American option's price is always $\ge$ intrinsic value. Bug 3 can violate this lower bound.
</details>

<details><summary><b>E3</b></summary>

The true continuation value $f_n(x) = \mathbb E[\text{future cashflow} \mid X = x]$ is **non-linear** in $x$ — it's a smooth concave function for puts. A pure linear fit $[1, X]$ misses the curvature. $[1, X, X^2]$ captures the bend. Higher-order polynomials work too but add variance to the fit; quadratic is the LSM default.
</details>

<details><summary><b>E4</b></summary>

- **Good** because: we only care about $f_n$ where it influences the exercise decision, which is only on the ITM region. OTM points have no exercise benefit and including them pulls the fit toward zero in the wrong places.
- **Backfires** when: very few paths are ITM (deep OTM strike, short time horizon), making the regression noisy. With $< 3$ ITM paths, you cannot fit 3 coefficients reliably — fallback to a constant or skip exercise.
</details>

<details><summary><b>E5</b></summary>

If the OLS fit produces **negative** `estimatedContinuationValue` on some paths (which is possible for puts because the polynomial fit is unconstrained), then `exerciseValue >= estCont` can be true even when `exerciseValue < 0` (i.e., the put is OTM). Without the `np.maximum(., 0)`, you could "exercise" an OTM put with a negative exercise value, losing money.

Adding `np.maximum(., 0)` forces us to compare against **at least 0** — equivalent to saying "even if continuation looks negative, treat it as zero, because the holder could simply not exercise and not lose anything."
</details>


## 7. Cheat sheet (one screen)

```
INITIALIZE
    payoffDiscounted = max(0, K - paths[:, -1])

LOOP n = N-1, N-2, ..., 1:
    continuationPayoffDiscounted = exp(-r*dt) * payoffDiscounted
    X = paths[:, n-1]
    exerciseValue = K - X

    basis = stack([1, X, X**2], axis=1)
    mask = exerciseValue > 0
    coeffs = lstsq(basis[mask], continuationPayoffDiscounted[mask])[0]
    estimatedContinuationValue = basis @ coeffs

    # COMPARE TO FITTED CURVE, NOT NOISY REALIZATION
    whichPathsToExercise = (exerciseValue >= maximum(estContVal, 0)) & mask

    payoffDiscounted = where(whichPathsToExercise,
                             exerciseValue,
                             continuationPayoffDiscounted)

TIME ZERO
    putprice = max(K - S0,
                   exp(-r*dt) * payoffDiscounted.mean())  # don't forget the max!
```

### The 3 Practice-Final P5 bugs in one line each
1. **Drift**: `r*dt` → `(r - 0.5*sigma**2)*dt`
2. **Compare to fitted**: `continuationPayoffDiscounted` → `estimatedContinuationValue`
3. **Time-0 max**: add `max(K - S0, ...)` around the final mean

### Why does this work? (one paragraph)

We do **dynamic programming** backwards in time. At each step, the unknown is the conditional expectation of future cashflow, $\mathbb E[\text{future} \mid X]$. The LSM idea: estimate this with **OLS regression** of the simulated future cashflows on basis functions of $X$. This converts a hard conditional expectation into a one-line `lstsq`. Then act on the fitted estimate, propagate the resulting cashflows back, repeat. At $t = 0$ no regression is needed because we have only one starting state, so just take the average — but **don't forget that the holder can exercise at $t=0$** too.


## Appendix — notation glossary (only if you have time)

You'll see this notation in lecture slides and textbooks. Memorizing it is **not** required for the exam — the code-level intuition above is enough.

- $X_t$ = state variable (typically stock price)
- $Y_t$ = discounted exercise payoff at time $t$ (= `exerciseValue * exp(-r*t)`)
- $\tau_n$ = optimal stopping time on $\{t_n, \ldots, t_N\}$ (a random exercise time)
- $f_n(x) := \mathbb E[Y_{\tau_{n+1}} \mid X_{t_n} = x]$ = true continuation value function
- $\hat f_n$ = OLS regression estimate of $f_n$ (= `estimatedContinuationValue`)
- $\hat f_n^+(x) := \max(\hat f_n(x), 0)$ = clipped fit for non-negative continuation values

### Translation table — exam code ↔ slide notation

| Exam code (Python) | Slide notation |
|---|---|
| `paths[m, n]` | $X^{(m)}_{t_{n+1}}$ |
| `exerciseValue` | $K - X_{t_n}$ |
| `continuationPayoffDiscounted` | $e^{-r\Delta t} Y_{\hat\tau_{n+1}}$ |
| `estimatedContinuationValue` | $\hat f_n(X_{t_n})$ |
| `whichPathsToExercise` | $1\{Y_{t_n} \ge \hat f_n(X_{t_n})\}$ |
| `payoffDiscounted` (after step) | $Y_{\hat\tau_n}$ |
| `putprice` | $\widehat C^{LS}_M$ |

If a textbook problem uses Greek-letter notation, just translate to the Python on the right.
